In [1]:
# pip install networkx matplotlib numpy powerlaw tqdm

import json
import math
import os
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

try:
    import powerlaw
    HAS_POWERLAW = True
except ImportError:
    HAS_POWERLAW = False


INPUT_FILE = "sample_10pct.json"
OUT_DIR = Path("graph_analysis_outputs")
OUT_DIR.mkdir(exist_ok=True)


def load_graph_from_sample_json(path: str) -> tuple[nx.DiGraph, list[dict]]:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # supports either:
    # 1) top-level list
    # 2) {"root": [...]}
    if isinstance(data, dict) and "root" in data:
        papers = data["root"]
    else:
        papers = data

    G = nx.DiGraph()

    for paper in papers:
        pid = paper["id"]
        G.add_node(pid)

    for paper in papers:
        src = paper["id"]
        for ref in paper.get("references", []):
            # keep only internal refs if node exists in sampled graph
            if ref in G:
                G.add_edge(src, ref)

    return G, papers


def safe_mean(arr):
    return float(np.mean(arr)) if len(arr) else 0.0


def safe_median(arr):
    return float(np.median(arr)) if len(arr) else 0.0


def degree_stats(G: nx.DiGraph):
    in_deg = dict(G.in_degree())
    out_deg = dict(G.out_degree())
    total_deg = dict(G.degree())

    in_vals = np.array(list(in_deg.values()))
    out_vals = np.array(list(out_deg.values()))
    total_vals = np.array(list(total_deg.values()))

    isolates = [n for n, d in total_deg.items() if d == 0]

    max_in_node = max(in_deg, key=in_deg.get) if in_deg else None
    min_in_node = min(in_deg, key=in_deg.get) if in_deg else None
    max_out_node = max(out_deg, key=out_deg.get) if out_deg else None
    min_out_node = min(out_deg, key=out_deg.get) if out_deg else None
    max_total_node = max(total_deg, key=total_deg.get) if total_deg else None
    min_total_node = min(total_deg, key=total_deg.get) if total_deg else None

    stats = {
        "num_nodes": G.number_of_nodes(),
        "num_edges": G.number_of_edges(),
        "density": nx.density(G),
        "num_self_loops": nx.number_of_selfloops(G),
        "num_isolates": len(isolates),
        "fraction_isolates": len(isolates) / G.number_of_nodes() if G.number_of_nodes() else 0.0,
        "avg_in_degree": safe_mean(in_vals),
        "avg_out_degree": safe_mean(out_vals),
        "avg_total_degree": safe_mean(total_vals),
        "median_in_degree": safe_median(in_vals),
        "median_out_degree": safe_median(out_vals),
        "median_total_degree": safe_median(total_vals),
        "max_in_degree": int(in_deg[max_in_node]) if max_in_node is not None else None,
        "max_in_degree_node": int(max_in_node) if max_in_node is not None else None,
        "min_in_degree": int(in_deg[min_in_node]) if min_in_node is not None else None,
        "min_in_degree_node": int(min_in_node) if min_in_node is not None else None,
        "max_out_degree": int(out_deg[max_out_node]) if max_out_node is not None else None,
        "max_out_degree_node": int(max_out_node) if max_out_node is not None else None,
        "min_out_degree": int(out_deg[min_out_node]) if min_out_node is not None else None,
        "min_out_degree_node": int(min_out_node) if min_out_node is not None else None,
        "max_total_degree": int(total_deg[max_total_node]) if max_total_node is not None else None,
        "max_total_degree_node": int(max_total_node) if max_total_node is not None else None,
        "min_total_degree": int(total_deg[min_total_node]) if min_total_node is not None else None,
        "min_total_degree_node": int(min_total_node) if min_total_node is not None else None,
    }

    return stats, in_deg, out_deg, total_deg, isolates


def giant_components_info(G: nx.DiGraph):
    weak_components = sorted(nx.weakly_connected_components(G), key=len, reverse=True)
    strong_components = sorted(nx.strongly_connected_components(G), key=len, reverse=True)

    giant_weak_nodes = weak_components[0] if weak_components else set()
    giant_strong_nodes = strong_components[0] if strong_components else set()

    Gw = G.subgraph(giant_weak_nodes).copy()
    Gs = G.subgraph(giant_strong_nodes).copy()

    info = {
        "num_weakly_connected_components": len(weak_components),
        "num_strongly_connected_components": len(strong_components),
        "largest_weak_component_size": len(giant_weak_nodes),
        "largest_strong_component_size": len(giant_strong_nodes),
        "largest_weak_component_fraction": len(giant_weak_nodes) / G.number_of_nodes() if G.number_of_nodes() else 0.0,
        "largest_strong_component_fraction": len(giant_strong_nodes) / G.number_of_nodes() if G.number_of_nodes() else 0.0,
    }
    return info, weak_components, strong_components, Gw, Gs


def approximate_path_stats(Gw: nx.DiGraph):
    # Use undirected giant weak component for classical path-length style analysis
    U = Gw.to_undirected()
    result = {}

    if U.number_of_nodes() == 0:
        return {
            "avg_clustering_undirected_giant_weak": None,
            "approx_diameter_undirected_giant_weak": None,
            "avg_shortest_path_sampled_pairs_undirected_giant_weak": None,
        }

    result["avg_clustering_undirected_giant_weak"] = float(nx.average_clustering(U))

    # Approx diameter via double sweep heuristic
    nodes = list(U.nodes())
    start = nodes[0]
    lengths1 = nx.single_source_shortest_path_length(U, start)
    far1 = max(lengths1, key=lengths1.get)

    lengths2 = nx.single_source_shortest_path_length(U, far1)
    far2 = max(lengths2, key=lengths2.get)
    result["approx_diameter_undirected_giant_weak"] = int(lengths2[far2])

    # Sampled average shortest path
    rng = np.random.default_rng(42)
    sample_nodes = rng.choice(nodes, size=min(200, len(nodes)), replace=False)
    dists = []
    for s in sample_nodes:
        sp = nx.single_source_shortest_path_length(U, s)
        dists.extend(sp.values())

    result["avg_shortest_path_sampled_pairs_undirected_giant_weak"] = float(np.mean(dists)) if dists else None
    return result


def reciprocity_and_assortativity(G: nx.DiGraph):
    out = {}
    try:
        out["reciprocity"] = float(nx.reciprocity(G)) if G.number_of_edges() else None
    except Exception:
        out["reciprocity"] = None

    try:
        # on directed graphs networkx may still return a scalar degree assortativity
        out["degree_assortativity"] = float(nx.degree_assortativity_coefficient(G))
    except Exception:
        out["degree_assortativity"] = None

    return out


def top_k_nodes_by_degree(in_deg, out_deg, total_deg, k=20):
    top_in = sorted(in_deg.items(), key=lambda x: x[1], reverse=True)[:k]
    top_out = sorted(out_deg.items(), key=lambda x: x[1], reverse=True)[:k]
    top_total = sorted(total_deg.items(), key=lambda x: x[1], reverse=True)[:k]
    return {
        "top_in_degree": [{"node": int(n), "degree": int(d)} for n, d in top_in],
        "top_out_degree": [{"node": int(n), "degree": int(d)} for n, d in top_out],
        "top_total_degree": [{"node": int(n), "degree": int(d)} for n, d in top_total],
    }


def plot_degree_histograms(in_deg, out_deg, total_deg):
    deg_sets = {
        "in_degree": np.array(list(in_deg.values())),
        "out_degree": np.array(list(out_deg.values())),
        "total_degree": np.array(list(total_deg.values())),
    }

    for name, vals in deg_sets.items():
        vals_pos = vals[vals > 0]
        if len(vals_pos) == 0:
            continue

        plt.figure(figsize=(8, 5))
        bins = np.logspace(np.log10(max(1, vals_pos.min())), np.log10(vals_pos.max()), 50)
        plt.hist(vals_pos, bins=bins)
        plt.xscale("log")
        plt.yscale("log")
        plt.xlabel(name)
        plt.ylabel("Count")
        plt.title(f"{name} distribution (log-log histogram)")
        plt.tight_layout()
        plt.savefig(OUT_DIR / f"{name}_hist_loglog.png", dpi=200)
        plt.close()


def ccdf(values):
    values = np.array(values)
    values = values[values > 0]
    if len(values) == 0:
        return np.array([]), np.array([])

    counts = Counter(values)
    xs = np.array(sorted(counts.keys()))
    freqs = np.array([counts[x] for x in xs])
    pdf = freqs / freqs.sum()
    ccdf_vals = 1 - np.cumsum(pdf) + pdf
    return xs, ccdf_vals


def plot_degree_ccdfs(in_deg, out_deg, total_deg):
    deg_sets = {
        "in_degree": list(in_deg.values()),
        "out_degree": list(out_deg.values()),
        "total_degree": list(total_deg.values()),
    }

    plt.figure(figsize=(8, 5))
    for name, vals in deg_sets.items():
        xs, ys = ccdf(vals)
        if len(xs):
            plt.plot(xs, ys, marker="o", linestyle="none", markersize=3, label=name)

    plt.xscale("log")
    plt.yscale("log")
    plt.xlabel("Degree k")
    plt.ylabel("P(K ≥ k)")
    plt.title("Degree CCDFs (log-log)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / "degree_ccdfs_loglog.png", dpi=200)
    plt.close()


def plot_component_sizes(weak_components):
    sizes = [len(c) for c in weak_components]
    if not sizes:
        return

    plt.figure(figsize=(8, 5))
    plt.hist(sizes, bins=50)
    plt.yscale("log")
    plt.xlabel("Weakly connected component size")
    plt.ylabel("Count")
    plt.title("Weak component size distribution")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "weak_component_size_hist.png", dpi=200)
    plt.close()


def plot_year_distribution(papers):
    years = []
    for p in papers:
        y = p.get("year")
        if isinstance(y, (int, float)):
            years.append(int(y))

    if not years:
        return

    plt.figure(figsize=(10, 5))
    plt.hist(years, bins=50)
    plt.xlabel("Year")
    plt.ylabel("Count")
    plt.title("Paper year distribution")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "year_distribution.png", dpi=200)
    plt.close()


def plot_n_citation_distribution(papers):
    vals = []
    for p in papers:
        c = p.get("n_citation")
        if isinstance(c, (int, float)) and c >= 0:
            vals.append(c)

    vals = np.array(vals)
    vals = vals[vals > 0]
    if len(vals) == 0:
        return

    plt.figure(figsize=(8, 5))
    bins = np.logspace(np.log10(max(1, vals.min())), np.log10(vals.max()), 50)
    plt.hist(vals, bins=bins)
    plt.xscale("log")
    plt.yscale("log")
    plt.xlabel("n_citation")
    plt.ylabel("Count")
    plt.title("n_citation distribution (log-log histogram)")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "n_citation_distribution_loglog.png", dpi=200)
    plt.close()


def draw_small_graph_preview(Gw: nx.DiGraph, max_nodes=500):
    if Gw.number_of_nodes() == 0:
        return

    # show a hub-heavy preview rather than a random preview
    top_nodes = sorted(Gw.degree, key=lambda x: x[1], reverse=True)[:max_nodes]
    keep = [n for n, _ in top_nodes]
    H = Gw.subgraph(keep).copy()

    plt.figure(figsize=(12, 10))
    pos = nx.spring_layout(H, seed=42, k=0.2)
    node_sizes = [max(20, 5 * H.degree(n)) for n in H.nodes()]
    nx.draw_networkx_nodes(H, pos, node_size=node_sizes, alpha=0.7)
    nx.draw_networkx_edges(H, pos, alpha=0.2, arrows=False)
    plt.title("Preview of high-degree region in giant weak component")
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "graph_preview_hubs.png", dpi=200)
    plt.close()


def fit_scale_free(total_deg):
    values = np.array(list(total_deg.values()))
    values = values[values > 0]

    result = {
        "powerlaw_package_available": HAS_POWERLAW,
        "note": "On a sampled induced subgraph, this is only approximate."
    }

    if len(values) == 0:
        result["status"] = "No positive degrees to fit."
        return result

    if HAS_POWERLAW:
        fit = powerlaw.Fit(values, discrete=True, verbose=False)
        alpha = float(fit.power_law.alpha)
        xmin = float(fit.power_law.xmin)

        R_pl_ln, p_pl_ln = fit.distribution_compare("power_law", "lognormal")
        R_pl_exp, p_pl_exp = fit.distribution_compare("power_law", "exponential")

        result.update({
            "status": "Fit completed with powerlaw.",
            "alpha": alpha,
            "xmin": xmin,
            "R_powerlaw_vs_lognormal": float(R_pl_ln),
            "p_powerlaw_vs_lognormal": float(p_pl_ln),
            "R_powerlaw_vs_exponential": float(R_pl_exp),
            "p_powerlaw_vs_exponential": float(p_pl_exp),
        })

        # plot fit
        plt.figure(figsize=(8, 5))
        fit.plot_ccdf(label="Empirical")
        fit.power_law.plot_ccdf(label=f"Power law fit (alpha={alpha:.3f})")
        plt.xlabel("Degree k")
        plt.ylabel("P(K ≥ k)")
        plt.title("Power-law fit on total degree CCDF")
        plt.legend()
        plt.tight_layout()
        plt.savefig(OUT_DIR / "powerlaw_fit_total_degree.png", dpi=200)
        plt.close()
    else:
        # fallback: crude tail regression on CCDF in log-log
        xs, ys = ccdf(values)
        mask = (xs >= np.percentile(xs, 80)) & (ys > 0)
        if mask.sum() >= 2:
            logx = np.log10(xs[mask])
            logy = np.log10(ys[mask])
            slope, intercept = np.polyfit(logx, logy, 1)
            result.update({
                "status": "Fallback tail regression used; install powerlaw for proper test.",
                "tail_loglog_slope": float(slope),
                "tail_loglog_intercept": float(intercept),
            })
        else:
            result["status"] = "Not enough tail points for fallback fit."

    return result


def main():
    print("Loading graph...")
    G, papers = load_graph_from_sample_json(INPUT_FILE)

    print("Computing basic stats...")
    basic_stats, in_deg, out_deg, total_deg, isolates = degree_stats(G)

    print("Computing component stats...")
    comp_info, weak_components, strong_components, Gw, Gs = giant_components_info(G)

    print("Computing path/clustering stats...")
    path_stats = approximate_path_stats(Gw)

    print("Computing reciprocity/assortativity...")
    rel_stats = reciprocity_and_assortativity(G)

    print("Computing top nodes...")
    top_nodes = top_k_nodes_by_degree(in_deg, out_deg, total_deg, k=20)

    print("Fitting scale-free model...")
    sf_stats = fit_scale_free(total_deg)

    print("Making plots...")
    plot_degree_histograms(in_deg, out_deg, total_deg)
    plot_degree_ccdfs(in_deg, out_deg, total_deg)
    plot_component_sizes(weak_components)
    plot_year_distribution(papers)
    plot_n_citation_distribution(papers)
    draw_small_graph_preview(Gw, max_nodes=500)

    summary = {}
    summary.update(basic_stats)
    summary.update(comp_info)
    summary.update(path_stats)
    summary.update(rel_stats)
    summary["top_nodes"] = top_nodes
    summary["scale_free_analysis"] = sf_stats

    with open(OUT_DIR / "summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print("\n===== SUMMARY =====")
    print(f"Nodes: {summary['num_nodes']}")
    print(f"Edges: {summary['num_edges']}")
    print(f"Density: {summary['density']:.8f}")
    print(f"Isolates: {summary['num_isolates']} ({100*summary['fraction_isolates']:.2f}%)")
    print(f"Max in-degree: {summary['max_in_degree']} at node {summary['max_in_degree_node']}")
    print(f"Max out-degree: {summary['max_out_degree']} at node {summary['max_out_degree_node']}")
    print(f"Max total degree: {summary['max_total_degree']} at node {summary['max_total_degree_node']}")
    print(f"Largest weak component: {summary['largest_weak_component_size']} nodes")
    print(f"Largest strong component: {summary['largest_strong_component_size']} nodes")
    print(f"Approx giant-component diameter: {summary['approx_diameter_undirected_giant_weak']}")
    print(f"Avg clustering (giant weak, undirected): {summary['avg_clustering_undirected_giant_weak']:.6f}")

    if HAS_POWERLAW:
        s = summary["scale_free_analysis"]
        print(f"Power-law alpha: {s.get('alpha')}")
        print(f"xmin: {s.get('xmin')}")
        print(f"PL vs lognormal: R={s.get('R_powerlaw_vs_lognormal')}, p={s.get('p_powerlaw_vs_lognormal')}")
        print(f"PL vs exponential: R={s.get('R_powerlaw_vs_exponential')}, p={s.get('p_powerlaw_vs_exponential')}")
    else:
        print("Install 'powerlaw' for a proper scale-free comparison.")

    print(f"\nSaved outputs in: {OUT_DIR.resolve()}")


if __name__ == "__main__":
    main()

Loading graph...
Computing basic stats...
Computing component stats...
Computing path/clustering stats...
Computing reciprocity/assortativity...
Computing top nodes...
Fitting scale-free model...


c:\Users\vikra\OneDrive\Desktop\NS-Project\.venv\Lib\site-packages\powerlaw\distributions.py:776: UserWarning: No valid fits found for distribution lognormal.
  warnings.warn(f"No valid fits found for distribution {self.name}.")


Making plots...

===== SUMMARY =====
Nodes: 490248
Edges: 462199
Density: 0.00000192
Isolates: 212421 (43.33%)
Max in-degree: 1377 at node 2161969291
Max out-degree: 96 at node 2620342231
Max total degree: 1379 at node 2161969291
Largest weak component: 248598 nodes
Largest strong component: 8 nodes
Approx giant-component diameter: 30
Avg clustering (giant weak, undirected): 0.093592
Power-law alpha: 2.7024563440675577
xmin: 18.0
PL vs lognormal: R=-0.05619064159281262, p=0.8376377770710401
PL vs exponential: R=890.8477058884006, p=1.801044323637504e-21

Saved outputs in: C:\Users\vikra\OneDrive\Desktop\NS-Project\graph_analysis_outputs
